In [ ]:
# The Spark Connect client must match the server's Spark version (apache/sedona runs Spark 4.0.1).
# A newer client (e.g. 4.2) requests configs the 4.0 server doesn't have and createDataFrame fails.
%pip install -q "pyspark[connect]==4.0.1" geopandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pyspark.sql import SparkSession
import geopandas as gpd
import pandas as pd

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
print(spark.version)

4.0.1


In [85]:
# List tables in the current active database
tables = spark.catalog.listTables()

# Print the names of the tables
for table in tables:
    print(table.name)
    
    df = spark.read.table(table.name)
    df.show()

places
+--------+----------+--------------------+
|    name|population|            geometry|
+--------+----------+--------------------+
|Cheyenne|     65000|POINT (-104.82 41...|
|Santa Fe|     88000|POINT (-105.94 35...|
| Boulder|    108000|POINT (-105.27 40...|
|  Denver|    715000|POINT (-104.99 39...|
+--------+----------+--------------------+

regions
+--------+--------------------+
|  region|            geometry|
+--------+--------------------+
|Colorado|POLYGON ((-109.05...|
| Wyoming|POLYGON ((-111.05...|
+--------+--------------------+



## Query the tables

In [86]:
# Spatial join: which region contains each place
spark.sql("""
    SELECT p.name, p.population, r.region, ST_AsText(p.geometry) AS wkt
    FROM places p
    JOIN regions r ON ST_Contains(r.geometry, p.geometry)
    ORDER BY p.population DESC
""").show(truncate=False)

# Great-circle distance (km) from Denver to every other place
spark.sql("""
    SELECT b.name, ROUND(ST_DistanceSphere(a.geometry, b.geometry) / 1000, 1) AS km_from_denver
    FROM places a
    JOIN places b ON a.name = 'Denver' AND b.name <> 'Denver'
    ORDER BY km_from_denver
""").show()

+--------+----------+--------+---------------------+
|name    |population|region  |wkt                  |
+--------+----------+--------+---------------------+
|Denver  |715000    |Colorado|POINT (-104.99 39.74)|
|Boulder |108000    |Colorado|POINT (-105.27 40.01)|
|Cheyenne|65000     |Wyoming |POINT (-104.82 41.14)|
+--------+----------+--------+---------------------+

+--------+--------------+
|    name|km_from_denver|
+--------+--------------+
| Boulder|          38.4|
|Cheyenne|         156.3|
|Santa Fe|         458.0|
+--------+--------------+



## Bring results back to GeoPandas

In [87]:
result = spark.sql("""
    SELECT p.name, r.region, ST_AsBinary(p.geometry) AS geometry
    FROM places p
    JOIN regions r ON ST_Contains(r.geometry, p.geometry)
""").toPandas()

result_gdf = pd.DataFrame(
    result.drop(columns="geometry"),
)

result_gdf.to_json('output/contains.json', orient='records')

result_gdf

,name,region
0,Cheyenne,Wyoming
1,Boulder,Colorado
2,Denver,Colorado


In [88]:
spark.stop()